# Paper Reproduction — Comparative Allocation Harness

Full-sample (2003–2026, 29 assets) reproduction of the master comparison table and statistical tests.

In [ ]:
import sys
from pathlib import Path
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
pd.set_option("display.max_rows", None)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from _shared import (
    ROOT, FAMILY_COLORS, FAMILY_MAP, FAMILY_ORDER, DISPLAY,
    load_base_returns, load_vmp_returns, load_regime_signals,
    build_all_stats, ann_sharpe, ann_return, ann_vol, max_drawdown
)

base = load_base_returns()
vmp  = load_vmp_returns()
sw_oos = pd.read_parquet(ROOT / "data/cache/portfolio_returns/switch_v2a_oos_29assets.parquet")
canon = pd.read_csv(ROOT / "data/cache/appendix_a_canonical.csv")

switch_v2a = sw_oos["SWITCH_v2a_train_only"].reindex(base.index)
switch_v2a.name = "SWITCH(v2a)"

print(f"Base returns: {base.shape}, VMP: {vmp.shape}")
print(f"Canon CSV: {len(canon)} rows")

## §3.1 Master Comparison Table (62 strategies)

In [ ]:
from IPython.display import display

# Build ordered index: for each strategy in FAMILY_ORDER, show base then VMP variant
ordered_rows = []
for strat in FAMILY_ORDER:
    base_match = canon[canon["strategy"] == strat]
    if len(base_match):
        ordered_rows.append(base_match.iloc[0])
    vmp_strat = f"VMP({strat})"
    vmp_match = canon[canon["strategy"] == vmp_strat]
    if len(vmp_match):
        ordered_rows.append(vmp_match.iloc[0])

# Add any remaining rows not covered
covered = {r["strategy"] for r in ordered_rows}
for _, row in canon.iterrows():
    if row["strategy"] not in covered:
        ordered_rows.append(row)

df_table = pd.DataFrame(ordered_rows).reset_index(drop=True)

# Select and rename display columns
cols_show = ["display", "family", "sharpe", "ann_ret", "ann_vol", "max_dd", "hit_pct", "net_10bps"]
col_rename = {
    "display": "Strategy",
    "family": "Family",
    "sharpe": "Sharpe",
    "ann_ret": "Ann Ret",
    "ann_vol": "Ann Vol",
    "max_dd": "Max DD",
    "hit_pct": "Hit%",
    "net_10bps": "Net 10bps",
}

df_display = df_table[cols_show].rename(columns=col_rename).copy()
for col in ["Sharpe", "Net 10bps"]:
    df_display[col] = pd.to_numeric(df_display[col], errors="coerce").round(3)
for col in ["Ann Ret", "Ann Vol", "Max DD"]:
    df_display[col] = pd.to_numeric(df_display[col], errors="coerce").map(lambda x: f"{x:.2%}" if pd.notna(x) else "")
df_display["Hit%"] = df_display["Hit%"].apply(lambda x: f"{float(x):.1f}" if pd.notna(x) and x != "—" and x != "" else "—")

display(df_display)

## §3.2 Rankings

In [ ]:
canon_num = canon.copy()
canon_num["sharpe"] = pd.to_numeric(canon_num["sharpe"], errors="coerce")
canon_num["ann_ret"] = pd.to_numeric(canon_num["ann_ret"], errors="coerce")
canon_num["net_10bps"] = pd.to_numeric(canon_num["net_10bps"], errors="coerce")

print("=== Top 10 by Gross Sharpe (all 62 strategies) ===")
top10_sharpe = canon_num.nlargest(10, "sharpe")[["display", "family", "sharpe", "ann_ret", "max_dd"]]
top10_sharpe["ann_ret"] = top10_sharpe["ann_ret"].map(lambda x: f"{x:.2%}")
top10_sharpe["max_dd"] = pd.to_numeric(top10_sharpe["max_dd"], errors="coerce").map(lambda x: f"{x:.2%}")
display(top10_sharpe.reset_index(drop=True))

print("\n=== Top 5 by Annualized Return ===")
top5_ret = canon_num.nlargest(5, "ann_ret")[["display", "family", "ann_ret", "sharpe"]]
top5_ret["ann_ret"] = top5_ret["ann_ret"].map(lambda x: f"{x:.2%}")
display(top5_ret.reset_index(drop=True))

print("\n=== Bottom 5 Base Strategies by Sharpe ===")
base_only = canon_num[~canon_num["is_vmp"]]
bot5 = base_only.nsmallest(5, "sharpe")[["display", "family", "sharpe", "ann_ret"]]
bot5["ann_ret"] = bot5["ann_ret"].map(lambda x: f"{x:.2%}")
display(bot5.reset_index(drop=True))

## §3.3 Transaction-Cost Sensitivity

In [ ]:
import matplotlib.patches as mpatches

canon_num2 = canon.copy()
for c in ["sharpe", "net_10bps", "net_strat", "turnover"]:
    canon_num2[c] = pd.to_numeric(canon_num2[c], errors="coerce")

print("=== Top 10 by Net Sharpe (10bps flat, artifact excluded) ===")
no_art = canon_num2[canon_num2["strategy"] != "VMP(GMV(sample))"]
top10_net = no_art.nlargest(10, "net_10bps")[["display", "sharpe", "net_10bps", "turnover"]]
display(top10_net.reset_index(drop=True))

print("\n=== Top 5 Degradation (sharpe - net_10bps, base only) ===")
base_only2 = canon_num2[~canon_num2["is_vmp"]].copy()
base_only2["degradation"] = base_only2["sharpe"] - base_only2["net_10bps"]
top5_deg = base_only2.nlargest(5, "degradation")[["display", "sharpe", "net_10bps", "turnover", "degradation"]]
display(top5_deg.reset_index(drop=True))

# Stratified vs flat scatter
fig, ax = plt.subplots(figsize=(8, 6))
for family, grp in canon_num2.groupby("family"):
    color = FAMILY_COLORS.get(family, "#aaaaaa")
    base_g = grp[~grp["is_vmp"]].dropna(subset=["net_10bps", "net_strat"])
    vmp_g  = grp[grp["is_vmp"]].dropna(subset=["net_10bps", "net_strat"])
    if len(base_g):
        ax.scatter(base_g["net_10bps"], base_g["net_strat"], color=color, s=40, label=family, zorder=4)
    if len(vmp_g):
        ax.scatter(vmp_g["net_10bps"], vmp_g["net_strat"], color="none", edgecolors=color,
                   s=60, linewidths=1.5, zorder=4)

lim_vals = canon_num2[["net_10bps", "net_strat"]].dropna()
lim = [lim_vals.min().min() - 0.05, lim_vals.max().max() + 0.05]
ax.plot(lim, lim, "k--", lw=0.8, alpha=0.5)
ax.set_xlabel("Net Sharpe — Flat 10bps")
ax.set_ylabel("Net Sharpe — Stratified Costs")
ax.set_title("Stratified vs. Flat Transaction Costs — All 62 Strategies", fontweight="bold")
patches = [mpatches.Patch(color=FAMILY_COLORS.get(f, "#aaa"), label=f) for f in sorted(canon_num2["family"].unique())]
ax.legend(handles=patches, ncol=2, fontsize=7, loc="upper left")
plt.tight_layout()
plt.show()

## §7.4 Sub-Period Sharpe (5-year buckets)

In [ ]:
BUCKETS = [
    ("2003–2007", "2003-01-01", "2007-12-31"),
    ("2008–2012", "2008-01-01", "2012-12-31"),
    ("2013–2017", "2013-01-01", "2017-12-31"),
    ("2018–2022", "2018-01-01", "2022-12-31"),
    ("2023–2026", "2023-01-01", "2026-12-31"),
]

REP_STRATS = [
    ("EW",                  base),
    ("GMV(ledoit_wolf)",    base),
    ("MSR(ledoit_wolf)",    base),
    ("MDP(ledoit_wolf)",    base),
    ("SWITCH(ledoit_wolf)", base),
    ("BL-Mom(LW)",          base),
    ("VMP(MDP(ledoit_wolf))", vmp),
]

# Add SWITCH(v2a) from sw_oos
sw_v2a_full = switch_v2a

rows = {}
for label, start, end in BUCKETS:
    col_sharpes = {}
    for col, src in REP_STRATS:
        r = src[col].loc[start:end].dropna() if col in src.columns else pd.Series(dtype=float)
        col_sharpes[col.replace("ledoit_wolf", "LW")] = ann_sharpe(r) if len(r) > 21 else np.nan
    # SWITCH(v2a)
    sv = sw_v2a_full.loc[start:end].dropna()
    col_sharpes["SWITCH(v2a)"] = ann_sharpe(sv) if len(sv) > 21 else np.nan
    rows[label] = col_sharpes

df_subperiod = pd.DataFrame(rows).T.round(3)
display(df_subperiod)

## Reproducibility Assertion

In [ ]:
TOL = 1e-4
failures = []

for _, row in canon.iterrows():
    strat = row["strategy"]
    canon_sharpe = pd.to_numeric(row["sharpe"], errors="coerce")
    if pd.isna(canon_sharpe):
        continue

    if strat in base.columns:
        computed = ann_sharpe(base[strat])
    elif strat in vmp.columns:
        computed = ann_sharpe(vmp[strat])
    else:
        continue

    if abs(computed - canon_sharpe) > TOL:
        failures.append((strat, canon_sharpe, computed, abs(computed - canon_sharpe)))

if failures:
    print(f"REPRODUCIBILITY FAIL — {len(failures)} mismatches:")
    for s, c, r, d in failures:
        print(f"  {s}: canon={c:.6f}, computed={r:.6f}, diff={d:.2e}")
else:
    print("Reproducibility: OK — all 62 rows match.")